In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = (
    SparkSession.builder
    .appName("TesteSpark")
    .master("spark://spark-master:7077")
    .getOrCreate()
)

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/24 00:00:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
dados = [
    (1, "João", 5000),
    (2, "Maria", 8000),
    (3, "Pedro", 3000)
]

df = spark.createDataFrame(
    dados,
    ["id_cliente", "nome", "renda"]
)

df.show()

+----------+-----+-----+
|id_cliente| nome|renda|
+----------+-----+-----+
|         1| João| 5000|
|         2|Maria| 8000|
|         3|Pedro| 3000|
+----------+-----+-----+



In [4]:
df.printSchema()

root
 |-- id_cliente: long (nullable = true)
 |-- nome: string (nullable = true)
 |-- renda: long (nullable = true)



In [5]:
df.count()

3

In [7]:
df_filtrado = df.filter(df.renda > 4000)

In [9]:
df_filtrado.explain(True)

== Parsed Logical Plan ==
'Filter (renda#2L > 4000)
+- LogicalRDD [id_cliente#0L, nome#1, renda#2L], false

== Analyzed Logical Plan ==
id_cliente: bigint, nome: string, renda: bigint
Filter (renda#2L > cast(4000 as bigint))
+- LogicalRDD [id_cliente#0L, nome#1, renda#2L], false

== Optimized Logical Plan ==
Filter (isnotnull(renda#2L) AND (renda#2L > 4000))
+- LogicalRDD [id_cliente#0L, nome#1, renda#2L], false

== Physical Plan ==
*(1) Filter (isnotnull(renda#2L) AND (renda#2L > 4000))
+- *(1) Scan ExistingRDD[id_cliente#0L,nome#1,renda#2L]



In [10]:
df.rdd.getNumPartitions()

2

In [11]:
df.repartition(4).rdd.getNumPartitions()

4

In [12]:
df.repartition(4).explain(True)

== Parsed Logical Plan ==
Repartition 4, true
+- LogicalRDD [id_cliente#0L, nome#1, renda#2L], false

== Analyzed Logical Plan ==
id_cliente: bigint, nome: string, renda: bigint
Repartition 4, true
+- LogicalRDD [id_cliente#0L, nome#1, renda#2L], false

== Optimized Logical Plan ==
Repartition 4, true
+- LogicalRDD [id_cliente#0L, nome#1, renda#2L], false

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Exchange RoundRobinPartitioning(4), REPARTITION_BY_NUM, [plan_id=94]
   +- Scan ExistingRDD[id_cliente#0L,nome#1,renda#2L]



In [13]:
spark.sparkContext.defaultParallelism

2

In [14]:
df.rdd.glom().collect()

[[Row(id_cliente=1, nome='João', renda=5000)],
 [Row(id_cliente=2, nome='Maria', renda=8000),
  Row(id_cliente=3, nome='Pedro', renda=3000)]]

In [4]:
clientes = []

for i in range(100000):

    clientes.append(

        (

            i,

            f"Cliente_{i}",

            (i % 15000) + 1000

        )

    )

df_clientes = spark.createDataFrame(

    clientes,

    ["id_cliente", "nome", "renda"]

)

In [5]:
df_clientes.count()

26/06/24 00:02:38 WARN TaskSetManager: Stage 0 contains a task of very large size (1173 KiB). The maximum recommended task size is 1000 KiB.


100000

In [6]:
df_clientes.groupBy("renda").count().show()

26/06/24 00:05:57 WARN TaskSetManager: Stage 3 contains a task of very large size (1173 KiB). The maximum recommended task size is 1000 KiB.


+-----+-----+
|renda|count|
+-----+-----+
| 1677|    7|
| 1697|    7|
| 1806|    7|
| 1950|    7|
| 2040|    7|
| 2214|    7|
| 2250|    7|
| 2453|    7|
| 2509|    7|
| 2529|    7|
| 2927|    7|
| 3091|    7|
| 3506|    7|
| 3764|    7|
| 4590|    7|
| 4823|    7|
| 4894|    7|
| 5385|    7|
| 5409|    7|
| 5556|    7|
+-----+-----+
only showing top 20 rows



In [9]:
spark.sparkContext.defaultParallelism

2